# Lecture 38: Wrap-up — Updating Probabilities
v.ekc

One more pass at Bayes' rule, standalone: students and majors, the medical test, and why your prior matters. If you can draw the tree, you can answer the question. (Slides: KL Lecture 38 deck.)

**Today's sections:**
1. Updating probabilities
2. The tree diagram
3. Decisions
4. Subjective probabilities

### 📋 Board Reference — updating a probability

| Piece | Meaning |
|---|---|
| **prior** | P(class) before seeing evidence |
| **likelihood** | P(evidence \| class) |
| tree branch | prior × likelihood |
| **posterior** | branch ÷ (sum of all branches with that evidence) |
| Bayes' rule | P(class \| evidence) = P(class & evidence) / P(evidence) |

In [ ]:
from datascience import *
import numpy as np
import matplotlib
from mpl_toolkits.mplot3d import Axes3D
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

---
## 1. Updating Probabilities

Build the population, then *count* — every conditional probability is just a ratio of counts.

In [ ]:
n = 100
second = round(n * 0.6)
third = round(n * 0.4)

year = np.array(['Second'] * second + ['Third'] * third)
major = np.array(['Declared'] * (round(second * 0.5)) + ['Undeclared'] * (round(second * 0.5)) + \
                 ['Declared'] * (round(third * 0.8))  + ['Undeclared'] * (round(third * 0.2)))
                 
students = Table().with_columns(
    'Year', year,
    'Major', major
)

In [ ]:
students.show(3)

In [ ]:
students.pivot('Major', 'Year')

In [ ]:
# Verify: 60% of students are Second years, 40% are Third years
60 / (60 + 40)

In [ ]:
# Verify: 50% of Second years have Declared
30 / 60

In [ ]:
# Verify: 80% of Third years have Declared
32 / 40

In [ ]:
# Chance of second year, given that they have declared
# P(second year | declared)

30 / 62

In [ ]:
# P(third year | declared)

32 / 62

---
## 2. The Tree Diagram

Same answer without building the population: multiply along branches, divide by the total.

In [ ]:
# P(second year | declared), from tree diagram

(0.6 * 0.5) / (0.6 * 0.5 + 0.4 * 0.8)

### Try It 1: Trees two ways 😊

1. Compute P(third year | declared) from the tree diagram, and check it against the pivot-table count (32/62).

2. Compute P(second year | **un**declared) both ways.

In [ ]:
# 1. P(third | declared), tree then count


In [ ]:
# 2. P(second | undeclared), tree then count


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Prior × likelihood on top, all matching branches on the bottom.*

```python
# 1.
(0.4 * 0.8) / (0.6 * 0.5 + 0.4 * 0.8)    # ≈ 0.516  (= 32/62)

# 2.
(0.6 * 0.5) / (0.6 * 0.5 + 0.4 * 0.2)    # ≈ 0.789  (= 30/38)
```

</details>

---
## 3. Decisions 🏥

The 1-in-1000 disease, the 99%-accurate test, and the positive result — count the population and see the base rate fallacy in action.

In [ ]:
def create_population(prior_disease_prob, n):
    disease = round(n * prior_disease_prob)
    no_disease = round(n * (1 - prior_disease_prob))

    status = np.array(['Disease'] * disease  +  ['No disease'] * no_disease)
    result = np.array(['Test +'] * (round(disease * 0.99)) + ['Test -'] * (round(disease * 0.01)) +\
                      ['Test +'] * (round(no_disease * 0.05)) + ['Test -'] * (round(no_disease * 0.95)) )
                 
    t = Table().with_columns(
    'Status', status,
    'Test Result', result
    )
    return t.pivot('Test Result', 'Status')

In [ ]:
create_population(1/1000, 100000)

In [ ]:
99 / (99 + 4995)

In [ ]:
#P(disease | tested +)

# = P(disease & tested +) / P(tested +)

(0.001 * 0.99) / (0.001*0.99 + 0.999*0.05)

> **≈ 2%, not 99%.** In 100,000 people: 99 true positives vs. 4,995 false positives — a positive test is overwhelmingly likely to be a false alarm when the disease is rare.

---
## 4. Subjective Probabilities

Change the prior (symptoms! family history!) and the same test result means something completely different:

In [ ]:
# P(disease | tested +)

# = P(disease & tested +) / P(tested +)

# if prior probability of disease is 1/10

(0.1 * 0.99) / (0.1*0.99 + 0.9*0.05)

In [ ]:
create_population(1/10, 10000)

In [ ]:
990/(990+450)

In [ ]:
# P(disease | tested +)
# if prior probability of disease is 0.5

(0.5 * 0.99) / (0.5*0.99 + 0.5*0.05)

In [ ]:
create_population(0.5, 10000)

In [ ]:
4950/(4950+250)

### Try It 2: Your own prior 😊

1. Suppose the prior is 1/100 (mild symptoms). What's P(disease | positive test)? Predict roughly, then compute both ways.

In [ ]:
# 1. prior = 1/100


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Ten times the 1/1000 prior → posterior jumps from ~2% to ~17%.*

```python
# 1.
(0.01 * 0.99) / (0.01 * 0.99 + 0.99 * 0.05)   # ≈ 0.167
create_population(1/100, 100000)                # count it too
```

</details>

---
> **Today's takeaway:** prior × likelihood, divided by all the ways the evidence happens — that's the whole trick. **Homework 12's condor vs. turkey vulture questions are this exact tree diagram**, with wingspans instead of test results. 🦅

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Bayes via tree diagram
```python
posterior = (prior * likelihood) / sum_of_all_branches_with_this_evidence
```